# Submissão B — DistilRoBERTa Fine-tuning

Classificação de texto (Human / Google / Meta / OpenAI / Mistral) usando fine-tuning do modelo `distilroberta-base` da HuggingFace.

## 1. Imports e configuração

In [ ]:
import os
import re
import time
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR
from transformers import AutoTokenizer, RobertaModel, RobertaConfig
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

MODEL_NAME     = 'distilroberta-base'  
MAX_LEN        = 128                    
BATCH_SIZE     = 16                    
EPOCHS         = 3
LR             = 2e-5
MAX_PER_CLASS  = 5000                 
SEED           = 42
CHECKPOINT_DIR = 'distilroberta_ckpt'  

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
torch.manual_seed(SEED)
np.random.seed(SEED)

c:\Users\Marco\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


## 2. Classes e labels

In [ ]:
TRAIN_CLASSES  = ['human', 'google', 'meta', 'openai', 'anthropic']

# Mapeamento label / índice
LABEL2IDX = {c: i for i, c in enumerate(TRAIN_CLASSES)}
IDX2LABEL = {i: c for c, i in LABEL2IDX.items()}

# Mapeamento para o formato de submissão 
SUBMISSION_MAP = {
    'human':     'Human',
    'google':    'Google',
    'meta':      'Meta',
    'openai':    'OpenAI',
    'anthropic': 'Anthropic',
}

N_CLASSES = len(TRAIN_CLASSES)
print(f'Classes ({N_CLASSES}): {TRAIN_CLASSES}')
print(f'LABEL2IDX: {LABEL2IDX}')

Classes (5): ['human', 'google', 'meta', 'openai', 'anthropic']
LABEL2IDX: {'human': 0, 'google': 1, 'meta': 2, 'openai': 3, 'anthropic': 4}


## 3. Carregar e preparar o dataset de treino

In [4]:
print('A carregar dataset...')
df_raw = load_dataset('BrunoFilipeRDS/dataset_2500_bal', split='train').to_pandas()
print(f'Total carregado: {len(df_raw)} linhas')
print(df_raw['label'].value_counts())

A carregar dataset...
Total carregado: 12500 linhas
label
anthropic    2500
openai       2500
google       2500
human        2500
meta         2500
Name: count, dtype: int64


In [ ]:
def clean_text(text):
    text = str(text)
    text = re.sub(r'<[^>]+>', ' ', text)          
    text = re.sub(r'\s+', ' ', text).strip()     
    return text

# Subsampling balanceado + limpeza
dfs = []
for label in TRAIN_CLASSES:
    subset = df_raw[df_raw['label'] == label].sample(
        n=min(MAX_PER_CLASS, len(df_raw[df_raw['label'] == label])),
        random_state=SEED
    )
    dfs.append(subset)

df = pd.concat(dfs, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
df['text']  = df['text'].apply(clean_text)
df['label_idx'] = df['label'].map(LABEL2IDX)

print(f'Dataset após sampling: {len(df)} linhas')
print(df['label'].value_counts())

Dataset após sampling: 12500 linhas
label
human        2500
anthropic    2500
openai       2500
google       2500
meta         2500
Name: count, dtype: int64


In [6]:
# Split treino / validação
df_train, df_val = train_test_split(
    df, test_size=0.1, random_state=SEED, stratify=df['label_idx']
)
print(f'Treino: {len(df_train)}  |  Validação: {len(df_val)}')

Treino: 11250  |  Validação: 1250


## 4. Tokenizer e Dataset PyTorch

In [ ]:
print(f'A carregar tokenizer: {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizer carregado.')

A carregar tokenizer: distilroberta-base...
Tokenizer carregado.


In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = list(texts)
        self.labels    = list(labels)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }


train_ds = TextDataset(df_train['text'].tolist(), df_train['label_idx'].tolist(), tokenizer, MAX_LEN)
val_ds   = TextDataset(df_val['text'].tolist(),   df_val['label_idx'].tolist(),   tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Batches treino: {len(train_loader)}  | Batches val: {len(val_loader)}')

Batches treino: 704  | Batches val: 79


## 5. Modelo: DistilRoBERTa + cabeça de classificação

In [ ]:
class DistilRoBERTaClassifier(nn.Module):
    def __init__(self, model_name, num_classes, dropout=0.3):
        super().__init__()
        self.encoder  = RobertaModel.from_pretrained(model_name)
        hidden_size   = self.encoder.config.hidden_size  
        self.dropout  = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        outputs    = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]   # token [CLS]
        return self.classifier(self.dropout(cls_output))


print(f'A carregar pesos pré-treinados de {MODEL_NAME}...')
model = DistilRoBERTaClassifier(MODEL_NAME, N_CLASSES).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parâmetros totais: {total_params:,}  |  Treináveis: {trainable:,}')

A carregar pesos pré-treinados de distilroberta-base...
Parâmetros totais: 82,316,549  |  Treináveis: 82,316,549


## 6. Treino

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = LinearLR(optimizer, start_factor=1.0, end_factor=0.1, total_iters=EPOCHS * len(train_loader))

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    t0 = time.time()

    for i, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['label'].to(DEVICE)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss   = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        train_loss    += loss.item() * labels.size(0)
        preds          = logits.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total   += labels.size(0)

        if (i + 1) % 50 == 0:
            elapsed = time.time() - t0
            print(f'  Epoch {epoch} | Batch {i+1}/{len(train_loader)} '
                  f'| Loss: {train_loss/train_total:.4f} '
                  f'| Acc: {train_correct/train_total:.4f} '
                  f'| {elapsed:.0f}s')

    train_acc  = train_correct / train_total
    train_loss /= train_total

    model.eval()
    val_correct, val_total, val_loss = 0, 0, 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in val_loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels         = batch['label'].to(DEVICE)

            logits     = model(input_ids, attention_mask)
            loss       = criterion(logits, labels)
            val_loss  += loss.item() * labels.size(0)
            preds      = logits.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total   += labels.size(0)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    val_acc  = val_correct / val_total
    val_loss /= val_total

    print(f'\n=== Epoch {epoch}/{EPOCHS} ===')
    print(f'  Treino  — Loss: {train_loss:.4f}  Acc: {train_acc:.4f}')
    print(f'  Val     — Loss: {val_loss:.4f}  Acc: {val_acc:.4f}')

    # Guardar o melhor modelo
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, 'best_model.pt'))
        tokenizer.save_pretrained(CHECKPOINT_DIR)
        # Guardar mapeamento de labels
        with open(os.path.join(CHECKPOINT_DIR, 'idx2label.pkl'), 'wb') as f:
            pickle.dump(IDX2LABEL, f)
        print(f'Novo melhor modelo guardado (val_acc={best_val_acc:.4f})')

print(f'\nTreino concluído. Melhor val_acc: {best_val_acc:.4f}')

  Epoch 1 | Batch 50/704 | Loss: 1.5225 | Acc: 0.3175 | 235s
  Epoch 1 | Batch 100/704 | Loss: 1.3013 | Acc: 0.4331 | 476s
  Epoch 1 | Batch 150/704 | Loss: 1.1481 | Acc: 0.4967 | 717s
  Epoch 1 | Batch 200/704 | Loss: 1.0512 | Acc: 0.5441 | 962s
  Epoch 1 | Batch 250/704 | Loss: 0.9855 | Acc: 0.5735 | 1192s
  Epoch 1 | Batch 300/704 | Loss: 0.9181 | Acc: 0.6048 | 1429s
  Epoch 1 | Batch 350/704 | Loss: 0.8690 | Acc: 0.6257 | 1672s
  Epoch 1 | Batch 400/704 | Loss: 0.8376 | Acc: 0.6389 | 1917s
  Epoch 1 | Batch 450/704 | Loss: 0.8055 | Acc: 0.6525 | 2147s
  Epoch 1 | Batch 500/704 | Loss: 0.7753 | Acc: 0.6663 | 2386s
  Epoch 1 | Batch 550/704 | Loss: 0.7519 | Acc: 0.6752 | 2625s
  Epoch 1 | Batch 600/704 | Loss: 0.7274 | Acc: 0.6865 | 2865s
  Epoch 1 | Batch 650/704 | Loss: 0.7077 | Acc: 0.6960 | 3098s
  Epoch 1 | Batch 700/704 | Loss: 0.6937 | Acc: 0.7024 | 3340s

=== Epoch 1/3 ===
  Treino  — Loss: 0.6925  Acc: 0.7031
  Val     — Loss: 0.4150  Acc: 0.8288
  ✓ Novo melhor modelo guard

In [11]:
# Relatório de classificação no set de validação (com o melhor modelo)
print('\nRelatório de classificação (validação):')
print(classification_report(
    all_labels, all_preds,
    target_names=TRAIN_CLASSES
))


Relatório de classificação (validação):
              precision    recall  f1-score   support

       human       0.87      0.61      0.72       250
      google       1.00      1.00      1.00       250
        meta       0.80      0.91      0.85       250
      openai       0.68      0.80      0.73       250
   anthropic       0.97      0.96      0.97       250

    accuracy                           0.85      1250
   macro avg       0.86      0.85      0.85      1250
weighted avg       0.86      0.85      0.85      1250



## 7. Inferência sobre `subm2.csv`

In [ ]:
print('A carregar modelo para inferência...')

with open(os.path.join(CHECKPOINT_DIR, 'idx2label.pkl'), 'rb') as f:
    IDX2LABEL_LOADED = pickle.load(f)

tokenizer_inf = AutoTokenizer.from_pretrained(CHECKPOINT_DIR)

model_inf = DistilRoBERTaClassifier(MODEL_NAME, N_CLASSES).to(DEVICE)
model_inf.load_state_dict(torch.load(
    os.path.join(CHECKPOINT_DIR, 'best_model.pt'),
    map_location=DEVICE
))
model_inf.eval()
print('Modelo carregado para inferência.')

A carregar modelo para inferência...
Modelo carregado para inferência.


In [ ]:
df_subm = pd.read_csv('subm2.csv', sep=';', encoding='utf-8-sig')
df_subm.columns = df_subm.columns.str.strip()
df_subm['Text'] = df_subm['Text'].astype(str).apply(clean_text)
print(f'Registos a classificar: {len(df_subm)}')
print(df_subm.head(3).to_string(index=False))

Registos a classificar: 150
    ID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   Text
D2-101                                                                       Microbial mats of coexisting bacteria and archaea were the dominant form of life 

In [ ]:
class InferenceDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len):
        self.texts     = list(texts)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0)
        }


inf_loader = DataLoader(
    InferenceDataset(df_subm['Text'].tolist(), tokenizer_inf, MAX_LEN),
    batch_size=BATCH_SIZE
)

all_preds_inf = []
with torch.no_grad():
    for batch in inf_loader:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        logits         = model_inf(input_ids, attention_mask)
        preds          = logits.argmax(dim=1).cpu().tolist()
        all_preds_inf.extend(preds)

labels_pred = [SUBMISSION_MAP[IDX2LABEL_LOADED[i]] for i in all_preds_inf]

print(f'Previsões geradas: {len(labels_pred)}')

Previsões geradas: 150


In [ ]:
df_out = pd.DataFrame({'ID': df_subm['ID'], 'Labels': labels_pred})
df_out.to_csv('subm2-g5-MECD-B.csv', index=False, sep=';')

print('Ficheiro guardado: subm2-g5-MECD-B.csv')
print('\nDistribuição de labels:')
print(df_out['Labels'].value_counts().to_string())
print('\nPrimeiras 10 linhas:')
print(df_out.head(10).to_string(index=False))

Ficheiro guardado: subm2-g5-MECD-B.csv

Distribuição de labels:
Labels
Anthropic    82
Meta         34
OpenAI       20
Human        14

Primeiras 10 linhas:
    ID    Labels
D2-101 Anthropic
D2-102 Anthropic
D2-103 Anthropic
D2-104 Anthropic
D2-105 Anthropic
D2-106 Anthropic
D2-107      Meta
D2-108 Anthropic
D2-109 Anthropic
D2-110    OpenAI
